# Trace-Bench × Harbor / Terminal-Bench 2.0 demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doxav/Trace-Bench/blob/harbor/notebooks/09_harbor_bench.ipynb)

This notebook demonstrates the new **Harbor benchmark adapter** in Trace-Bench.

It is designed to run in **Google Colab** from the `doxav/Trace-Bench` `harbor` branch and documents the full workflow:

1. install Trace-Bench from the `harbor` branch;
2. install Harbor CLI support;
3. read Colab secrets for `DAYTONA_API_KEY`, `OPENROUTER_API_KEY`, and `OPENAI_API_KEY`;
4. list Terminal-Bench tasks and Trace-Bench trainers;
5. build a Trace-Bench config for `terminal_bench:<task>`;
6. run a safe **stub** validation when keys are missing;
7. run a **real Harbor/Daytona** job when the required keys are available;
8. inspect `results.csv`, `summary.json`, `job_meta.json`, and Harbor artifacts such as trajectory files.

## Execution modes

- **Stub mode**: always runnable in Colab; Harbor is not invoked.
- **Real mode**: requires `DAYTONA_API_KEY` and either `OPENROUTER_API_KEY` or `OPENAI_API_KEY`.

The optimized object is the **Terminus-2 agent prompt template** exposed as a Trace trainable node by the Harbor adapter.


## 0. Runtime options

Edit these values before running if you want a different task, trainer, or model. The defaults are intentionally small so the notebook is suitable for a first validation pass.


In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, textwrap, time, glob

# --- Repository / notebook source ---
TRACE_BENCH_REPO = "https://github.com/doxav/Trace-Bench.git"
TRACE_BENCH_BRANCH = "harbor"
TRACE_BENCH_DIR = Path("/content/Trace-Bench")

# --- Harbor / Terminal-Bench task configuration ---
# Use a curated sample task from the adapter. Change after listing tasks below.
TASK_NAME = os.environ.get("TBENCH_TASK", "regex-log")
TRAINER_ID = os.environ.get("TRACE_BENCH_TRAINER", "PrioritySearch")
HARBOR_DATASET = os.environ.get("HARBOR_DATASET", "terminal-bench@2.0")

# Keep real runs bounded. Raise this only after a smoke run succeeds.
HARBOR_MAX_TURNS = int(os.environ.get("HARBOR_MAX_TURNS", "3"))
HARBOR_TIMEOUT_SECONDS = int(os.environ.get("HARBOR_TIMEOUT_SECONDS", "1800"))

# Model selection. Can be overridden via Colab secret/env HARBOR_MODEL or OPENROUTER_MODEL.
DEFAULT_OPENROUTER_MODEL = os.environ.get("OPENROUTER_MODEL", "openai/gpt-4o-mini")
DEFAULT_OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "openai/gpt-4o-mini")

# Where Trace-Bench run artifacts will be written.
RUN_DATE = time.strftime("%Y-%m-%d")
DEFAULT_DRIVE_RUNS = Path(f"/content/drive/MyDrive/bench/{RUN_DATE}/harbor_trace_bench")
DEFAULT_LOCAL_RUNS = Path(f"/content/harbor_trace_bench_runs/{RUN_DATE}")

print("Configured task:", TASK_NAME)
print("Configured trainer:", TRAINER_ID)
print("Harbor dataset:", HARBOR_DATASET)


## 1. Mount Google Drive for persistent artifacts

The notebook writes run outputs to Drive if available. If Drive mount is skipped or unavailable, it falls back to `/content`.


In [ ]:
IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
        RUNS_DIR = DEFAULT_DRIVE_RUNS
    except Exception as exc:
        print("Drive mount failed or was skipped; using local /content runs dir.")
        print(type(exc).__name__, exc)
        RUNS_DIR = DEFAULT_LOCAL_RUNS
else:
    RUNS_DIR = Path.cwd() / "runs" / "harbor_trace_bench"

RUNS_DIR.mkdir(parents=True, exist_ok=True)
os.environ["TRACE_BENCH_RUNS_DIR"] = str(RUNS_DIR)
print("RUNS_DIR =", RUNS_DIR)


## 2. Clone and install Trace-Bench from the `harbor` branch

Colab opens the notebook file, but it does not automatically clone the repository. This cell clones the target branch, checks out the exact code, and installs it in editable mode.


In [ ]:
def run_cmd(cmd, *, cwd=None, check=True, env=None):
    print("$", " ".join(map(str, cmd)))
    proc = subprocess.run(
        list(map(str, cmd)),
        cwd=str(cwd) if cwd else None,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(proc.stdout[-4000:])
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}: {' '.join(map(str, cmd))}")
    return proc

if IN_COLAB:
    if not TRACE_BENCH_DIR.exists():
        run_cmd(["git", "clone", "--depth", "1", "--branch", TRACE_BENCH_BRANCH, TRACE_BENCH_REPO, TRACE_BENCH_DIR])
    else:
        print(f"Repo already exists at {TRACE_BENCH_DIR}; refreshing {TRACE_BENCH_BRANCH}.")
        run_cmd(["git", "-C", TRACE_BENCH_DIR, "fetch", "origin", TRACE_BENCH_BRANCH, "--depth", "1"], check=False)
        run_cmd(["git", "-C", TRACE_BENCH_DIR, "checkout", TRACE_BENCH_BRANCH], check=False)
        run_cmd(["git", "-C", TRACE_BENCH_DIR, "pull", "--ff-only", "origin", TRACE_BENCH_BRANCH], check=False)
    os.chdir(TRACE_BENCH_DIR)
else:
    # Local notebook: assume it is being run from inside the repo or a checkout.
    if (Path.cwd() / "trace_bench").exists():
        TRACE_BENCH_DIR = Path.cwd()
    os.chdir(TRACE_BENCH_DIR)

print("CWD:", Path.cwd())
run_cmd(["git", "rev-parse", "--abbrev-ref", "HEAD"], check=False)
run_cmd(["git", "rev-parse", "HEAD"], check=False)

# Install Trace-Bench and lightweight notebook dependencies.
run_cmd([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"])
run_cmd([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
run_cmd([sys.executable, "-m", "pip", "install", "-q", "pandas", "pyyaml"])


## 3. Install Harbor CLI support

Real Harbor runs need a working `harbor` command. This cell tries common package names. If your environment requires a different Harbor install command, replace this cell with your internal install line.

Stub mode does **not** require Harbor.


In [ ]:
def install_harbor_best_effort():
    candidates = [
        [sys.executable, "-m", "pip", "install", "-q", "harbor[daytona]>=0.6.0"],
        [sys.executable, "-m", "pip", "install", "-q", "harbor>=0.6.0"],
    ]
    for cmd in candidates:
        proc = run_cmd(cmd, check=False)
        if proc.returncode == 0 and shutil.which("harbor"):
            return True
    return shutil.which("harbor") is not None

HARBOR_AVAILABLE = install_harbor_best_effort()
print("harbor available:", HARBOR_AVAILABLE, shutil.which("harbor"))
if HARBOR_AVAILABLE:
    run_cmd(["harbor", "--help"], check=False)
else:
    print("Harbor CLI not found. Real mode will be skipped; stub mode can still run.")


## 4. Load Colab secrets / environment keys

This cell uses Colab secrets if available:

- `DAYTONA_API_KEY` — required for real Harbor runs in Colab.
- `OPENROUTER_API_KEY` — preferred if present; exposed through OpenAI-compatible env vars.
- `OPENAI_API_KEY` — used if OpenRouter is not present.

If neither model key is set, the notebook automatically stays in **stub mode**.


In [ ]:
def get_secret_or_env(name: str) -> str:
    # Prefer existing env var, then Colab secret.
    if os.environ.get(name):
        return os.environ[name]
    if IN_COLAB:
        try:
            from google.colab import userdata  # type: ignore
            value = userdata.get(name)
            if value:
                return value
        except Exception:
            pass
    return ""

DAYTONA_API_KEY = get_secret_or_env("DAYTONA_API_KEY")
OPENROUTER_API_KEY = get_secret_or_env("OPENROUTER_API_KEY")
OPENAI_API_KEY = get_secret_or_env("OPENAI_API_KEY")

if DAYTONA_API_KEY:
    os.environ["DAYTONA_API_KEY"] = DAYTONA_API_KEY

# Provider policy:
# - Prefer OpenRouter when OPENROUTER_API_KEY is present.
# - Otherwise use OpenAI when OPENAI_API_KEY is present.
if OPENROUTER_API_KEY:
    os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
    os.environ["OPENAI_API_KEY"] = OPENROUTER_API_KEY
    os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"
    PROVIDER = "openrouter"
    HARBOR_MODEL = os.environ.get("HARBOR_MODEL") or DEFAULT_OPENROUTER_MODEL
elif OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    os.environ.pop("OPENAI_BASE_URL", None)
    PROVIDER = "openai"
    HARBOR_MODEL = os.environ.get("HARBOR_MODEL") or DEFAULT_OPENAI_MODEL
else:
    PROVIDER = "none"
    HARBOR_MODEL = os.environ.get("HARBOR_MODEL", "")

if HARBOR_MODEL:
    os.environ["HARBOR_MODEL"] = HARBOR_MODEL

REAL_READY = bool(DAYTONA_API_KEY and HARBOR_MODEL and (OPENROUTER_API_KEY or OPENAI_API_KEY) and HARBOR_AVAILABLE)
MODE = "real" if REAL_READY else "stub"
HARBOR_ENV = "daytona" if DAYTONA_API_KEY else "docker"

print("Provider:", PROVIDER)
print("DAYTONA_API_KEY set:", bool(DAYTONA_API_KEY))
print("OPENROUTER_API_KEY set:", bool(OPENROUTER_API_KEY))
print("OPENAI_API_KEY set:", bool(OPENAI_API_KEY if not OPENROUTER_API_KEY else OPENROUTER_API_KEY))
print("HARBOR_MODEL:", HARBOR_MODEL or "<not set>")
print("Harbor env:", HARBOR_ENV)
print("Selected Trace-Bench mode:", MODE)

if MODE == "stub":
    print("\nRunning in STUB mode: Harbor/Daytona will not be invoked. Add DAYTONA_API_KEY + model key for a real run.")


## 5. List Terminal-Bench tasks and Trace-Bench trainers

The Harbor adapter exposes a curated subset of Terminal-Bench 2.0 tasks for smoke/demo runs. You can still run any task by setting `TASK_NAME` to a valid Harbor task slug.


In [ ]:
print("Trace-Bench trainers:")
run_cmd([sys.executable, "-m", "trace_bench", "list-trainers", "--all"], check=False)

print("\nTerminal-Bench sample tasks:")
proc = run_cmd([sys.executable, "-m", "trace_bench", "list-tasks", "--bench", "terminal_bench"], check=False)
listed_tasks = [line.strip() for line in proc.stdout.splitlines() if line.strip().startswith("terminal_bench:")]
print("Parsed terminal_bench tasks:", listed_tasks)

if TASK_NAME not in [t.split(":", 1)[1] for t in listed_tasks] and listed_tasks:
    print(f"Configured TASK_NAME={TASK_NAME!r} is not in the curated list; this is allowed if Harbor knows it.")

print("\nChosen TASK_NAME:", TASK_NAME)
print("Chosen TRAINER_ID:", TRAINER_ID)


## 6. Create a Trace-Bench config

The config uses the same Trace-Bench runner surface as LLM4AD/VeriBench-style tasks:

- `tasks[].id = terminal_bench:<task>`
- Harbor parameters live under `tasks[].eval_kwargs`
- trainer parameters live under `trainers[].params_variants`

In **real mode**, this invokes Harbor with Daytona. In **stub mode**, Harbor is not invoked.


In [ ]:
import json

CONFIG_DIR = Path("configs")
CONFIG_DIR.mkdir(exist_ok=True)
CONFIG_PATH = CONFIG_DIR / "09_harbor_bench_colab.json"

config = {
    "runs_dir": str(RUNS_DIR),
    "mode": MODE,
    "seeds": [123],
    "max_workers": 1,
    "fail_fast": False,
    "tasks": [
        {
            "id": f"terminal_bench:{TASK_NAME}",
            "eval_kwargs": {
                "harbor_dataset": HARBOR_DATASET,
                "harbor_env": HARBOR_ENV,
                "harbor_model": HARBOR_MODEL or None,
                "harbor_cli_timeout_seconds": HARBOR_TIMEOUT_SECONDS,
                "harbor_agent_kwargs": {
                    "max_turns": HARBOR_MAX_TURNS,
                    "parser_name": "json",
                },
            },
        }
    ],
    "trainers": [
        {
            "id": TRAINER_ID,
            "params_variants": [
                {
                    "threads": 1,
                    "ps_steps": 1,
                    "ps_batches": 1,
                    "ps_candidates": 1,
                    "ps_proposals": 1,
                    "ps_mem_update": 1,
                }
            ],
            "optimizer_kwargs": {"memory_size": 5},
        }
    ],
}

CONFIG_PATH.write_text(json.dumps(config, indent=2), encoding="utf-8")
print("Wrote config:", CONFIG_PATH)
print(CONFIG_PATH.read_text())


## 7. Validate the config

Validation should succeed and write any validation artifacts under the same `RUNS_DIR`.


In [ ]:
validate_cmd = [
    sys.executable, "-m", "trace_bench", "validate",
    "--config", str(CONFIG_PATH),
    "--strict",
    "--runs-dir", str(RUNS_DIR),
]
proc = run_cmd(validate_cmd, check=False)
if proc.returncode != 0:
    raise RuntimeError("Validation failed. See output above.")


## 8. Run the Harbor benchmark through Trace-Bench

This uses `trace-bench run` exactly like the other benchmark integrations.


In [ ]:
run_cmd([
    sys.executable, "-m", "trace_bench", "run",
    "--config", str(CONFIG_PATH),
    "--runs-dir", str(RUNS_DIR),
], check=True)


## 9. Inspect run-level artifacts

The run should contain:

- `meta/config.snapshot.yaml` or `.json` depending on current writer
- `meta/manifest.json`
- `meta/env.json`
- `results.csv`
- `summary.json`
- `jobs/<job_id>/job_meta.json`
- `jobs/<job_id>/results.json`
- `jobs/<job_id>/events.jsonl`
- in real mode: Harbor artifacts under `jobs/<job_id>/artifacts/terminal_bench/`


In [ ]:
import pandas as pd


def find_run_dirs(root: Path):
    root = Path(root)
    dirs = []
    if root.exists():
        dirs.extend([p for p in root.iterdir() if p.is_dir() and (p / "results.csv").exists()])
        nested = root / "runs"
        if nested.exists():
            dirs.extend([p for p in nested.iterdir() if p.is_dir() and (p / "results.csv").exists()])
    return sorted(set(dirs), key=lambda p: p.stat().st_mtime)

run_dirs = find_run_dirs(RUNS_DIR)
assert run_dirs, f"No Trace-Bench run dirs found under {RUNS_DIR}"
latest_run = run_dirs[-1]
print("Latest run:", latest_run)
print("Files:", sorted([p.name for p in latest_run.iterdir()]))

summary_path = latest_run / "summary.json"
results_path = latest_run / "results.csv"
manifest_path = latest_run / "meta" / "manifest.json"

print("summary.json exists:", summary_path.exists())
print("results.csv exists:", results_path.exists())
print("manifest.json exists:", manifest_path.exists())

summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}
print("Summary:")
print(json.dumps(summary, indent=2)[:3000])

df = pd.read_csv(results_path)
display(df)


## 10. Inspect job-level artifacts and Harbor trajectory output

In stub mode, Harbor artifacts may be absent or minimal. In real mode, you should see Harbor job/trial outputs and a trajectory conversion file.


In [ ]:
jobs_dir = latest_run / "jobs"
job_dirs = sorted([p for p in jobs_dir.iterdir() if p.is_dir()])
assert job_dirs, "No job dirs found"

for job_dir in job_dirs:
    print("\n=== JOB", job_dir.name, "===")
    for rel in ["job_meta.json", "results.json", "events.jsonl"]:
        p = job_dir / rel
        print(rel, "exists:", p.exists())
        if p.exists() and p.stat().st_size < 12000:
            txt = p.read_text(encoding="utf-8", errors="replace")
            print(txt[:3000])

    tb_art = job_dir / "artifacts" / "terminal_bench"
    print("terminal_bench artifact dir:", tb_art, "exists:", tb_art.exists())
    if tb_art.exists():
        all_files = sorted([p for p in tb_art.rglob("*") if p.is_file()])
        print("Terminal-Bench artifact files:")
        for p in all_files[:50]:
            print(" -", p.relative_to(tb_art), f"({p.stat().st_size} bytes)")
        trajectory_files = [p for p in all_files if "trajectory" in p.name.lower()]
        if trajectory_files:
            print("\nTrajectory-related files:")
            for p in trajectory_files:
                print(" -", p)
            sample = trajectory_files[-1]
            print("\nSample trajectory artifact:", sample)
            print(sample.read_text(encoding="utf-8", errors="replace")[:4000])


## 11. Optional: switch task/trainer/model and rerun

To try a different task:

1. change `TASK_NAME` in Cell 0, or set `TBENCH_TASK` as an environment variable;
2. re-run cells 6 onward.

For real Harbor runs in Colab:

- add Colab secret `DAYTONA_API_KEY`;
- add either `OPENROUTER_API_KEY` or `OPENAI_API_KEY`;
- optionally set `HARBOR_MODEL` to a model string supported by your Harbor/LLM setup.

Suggested first real run settings:

```python
TASK_NAME = "regex-log"
HARBOR_MAX_TURNS = 3
HARBOR_TIMEOUT_SECONDS = 1800
```

Then re-run cells 6–10.


## 12. Validation checklist

After running all cells, check:

- [ ] `trace-bench list-tasks --bench terminal_bench` lists curated Harbor/Terminal-Bench tasks.
- [ ] `trace-bench list-trainers --all` lists available Trace trainers.
- [ ] `validate --strict --runs-dir <RUNS_DIR>` succeeds.
- [ ] `trace-bench run` writes a run under `RUNS_DIR`.
- [ ] `results.csv` contains `terminal_bench:<task>`.
- [ ] `summary.json` reports exactly one planned job for this demo config.
- [ ] In real mode, Harbor artifacts exist under `jobs/<job_id>/artifacts/terminal_bench/`.
- [ ] In real mode, trajectory artifacts exist and can be inspected.
